In [115]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from pydlm import dlm, trend, seasonality, dynamic
import pickle

In [116]:
df = pd.read_csv("../../data/pre_data.csv")

In [117]:
exog_cols = ["Thị_trường", "Loại_giá", "Nguồn"]

In [118]:
df.head()

,Ngày,Tên_mặt_hàng,Thị_trường,Loại_giá,Nguồn,Giá,Ngành_hàng
0,2020-01-01,24,19,7,2,32639.0,0
1,2020-01-01,24,12,7,2,32000.0,0
2,2020-01-02,24,19,7,2,32639.0,0
3,2020-01-02,24,12,7,2,32000.0,0
4,2020-01-03,24,19,7,2,32539.0,0


In [119]:
results_df = pd.DataFrame(columns=["Ngày", "Tên_mặt_hàng", "Giá", "Mean", "Var"])

In [120]:
for item in df["Tên_mặt_hàng"].unique():
    item_df = df[df["Tên_mặt_hàng"] == item]

    y = item_df["Giá"].values.tolist()
    X = item_df[exog_cols].values.tolist()

    model = dlm(y)
    model += trend(degree=1, discount=0.95)
    model += dynamic(X, discount=0.95, name="exog")
    model.fit()

    with open(f"../../results/dlm/{item}.pkl", "wb") as file:
        pickle.dump(model, file)

    features = {"exog": X}
    predict_mean, predict_var = model.predictN(N=len(item_df), date=model.n-1, featureDict=features)
    temp_df = pd.DataFrame({
        "Ngày": item_df["Ngày"],
        "Tên_mặt_hàng": item_df["Tên_mặt_hàng"],
        "Giá": item_df["Giá"],
        "Mean": predict_mean,
        "Var": predict_var
    })
    results_df = pd.concat([results_df, temp_df], axis=0)

INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
INFO:pydlm:Forward filtering completed.
INFO:pydlm:Starting backward smoothing...
INFO:pydlm:Backward smoothing completed.
C:\Users\Asriel\AppData\Local\Temp\ipykernel_15080\3451941199.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, temp_df], axis=0)
INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
INFO:pydlm:Forward filtering completed.
INFO:pydlm:Starting backward smoothing...
INFO:pydlm:Backward smoothing completed.
INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
IN

In [109]:
results_df.to_csv("../../results/dlm/forecast.csv", index=False)

In [110]:
h = len(item_df)
exog_future = item_df[exog_cols].values.tolist()
features = {"exog": exog_future}

predict_mean, predict_var = model.predictN(N=h, date=model.n-1, featureDict=features)

In [111]:
print(predict_mean)

[12689.976161722172, 12530.335958197938, 12370.695754673705, 12211.055551149471, 12051.415347625238, 11891.775144101004, 11732.13494057677, 11572.494737052537, 11412.854533528303, 11253.21433000407, 11093.574126479836, 10933.933922955603, 10774.293719431369]
